# Decision Tree Optimization Framework - M5 Pipeline (Google Colab, resumable)

**Before running:** the M5 CSV files must be in `My Drive/m5_data`.

Click **Runtime -> Run all**, allow Google Drive access, and keep this tab open.

This version is **resumable**: every finished step saves its results to `My Drive/m5_results`.
If Colab disconnects, just click **Run all** again - finished steps are loaded from Drive instead of being re-trained.
(Working files are kept on Colab's own disk and rebuilt in a few minutes when needed.)

Stores used: CA_1, TX_1, WI_1 (one per state).


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

!pip install -q lightgbm optuna cmaes shap pyarrow


In [ ]:
import os
import pandas as pd
import numpy as np
import lightgbm as lgb
import optuna
from optuna.samplers import TPESampler, CmaEsSampler
from optuna.pruners import MedianPruner, NopPruner
import warnings, gc, glob, shutil
warnings.filterwarnings("ignore")
optuna.logging.set_verbosity(optuna.logging.WARNING)

DATA_DIR = "/content/drive/MyDrive/m5_data"
OUT_DIR = "/content/drive/MyDrive/m5_results"
if not os.path.exists(f"{DATA_DIR}/sell_prices.csv"):
    raise FileNotFoundError("Put the M5 CSV files in My Drive/m5_data")
print("Files:", sorted(os.listdir(DATA_DIR)))

WORK_DIR = "/content/m5_work"            # big working files stay on Colab disk (fast, not lost by Drive sync)
MERGED_DIR = f"{WORK_DIR}/merged_by_store"
FEATS_STORE_DIR = f"{WORK_DIR}/features"
for d in [OUT_DIR, MERGED_DIR, FEATS_STORE_DIR]:
    os.makedirs(d, exist_ok=True)

store_list = ["CA_1", "TX_1", "WI_1"]    # one store per state

FORECAST_HORIZON = 7
TEST_DAYS = 90
TRAIN_WINDOW_DAYS = 365
VALID_DAYS = 56
N_TRIALS_BASELINE = 21                  # base paper's number of trials
N_TRIALS_OPTIMIZED = 40
N_TRIALS_JOINT_PER_STRATEGY = 20
PRUNE_CHECK_EVERY = 25
RANDOM_SEED = 42


## Step 1 - Data preparation (one store at a time, saved to disk)

In [ ]:
cal = pd.read_csv(f"{DATA_DIR}/calendar.csv", parse_dates=["date"])
if "d" not in cal.columns:
    cal["d"] = ["d_" + str(i + 1) for i in range(len(cal))]
prices = pd.read_csv(f"{DATA_DIR}/sell_prices.csv")

sales_path = f"{DATA_DIR}/sales_train_evaluation.csv"
id_cols = ["item_id", "dept_id", "cat_id", "store_id", "state_id"]
day_cols = [c for c in pd.read_csv(sales_path, nrows=0).columns if c.startswith("d_")]
print("Stores used:", store_list)

snap_cols = {"CA": "snap_CA", "TX": "snap_TX", "WI": "snap_WI"}
for store_id in store_list:
    out_path = f"{MERGED_DIR}/{store_id}.parquet"
    if os.path.exists(out_path):
        print(f"{store_id}: already prepared, skipping")
        continue
    parts = [ch[ch["store_id"] == store_id] for ch in pd.read_csv(sales_path, chunksize=2000)]
    sales_store = pd.concat(parts, ignore_index=True); del parts
    state = sales_store["state_id"].iloc[0]
    long_store = sales_store.melt(id_vars=id_cols, value_vars=day_cols, var_name="d", value_name="sales")
    long_store["sales"] = long_store["sales"].astype("float32")
    del sales_store; gc.collect()
    long_store = long_store.merge(
        cal[["d", "date", "wm_yr_wk", "wday", "event_name_1", "event_type_1", snap_cols[state]]],
        on="d", how="left").rename(columns={snap_cols[state]: "snap"})
    long_store = long_store.merge(prices, on=["store_id", "item_id", "wm_yr_wk"], how="left")
    long_store.to_parquet(out_path, index=False)
    print(f"{store_id}: saved {len(long_store):,} rows")
    del long_store; gc.collect()

max_date = cal["date"].max()
test_start = max_date - pd.Timedelta(days=TEST_DAYS - 1)
print(f"Test period: {test_start.date()} to {max_date.date()}")


## Step 2 - Feature engineering (base paper Appendix A, adapted to M5) and shared functions

In [ ]:
LAGS = list(range(1, 15))
WINDOWS = [7, 14, 30, 60, 180]
MANUAL_FEATURES = (
    [f"sales_lag_{l}" for l in LAGS]
    + [f"rolling_mean_{w}" for w in WINDOWS] + [f"rolling_stdv_{w}" for w in WINDOWS]
    + ["day_of_week", "day_of_month", "week_of_year", "month", "quarter", "end_of_week",
       "sell_price", "price_ratio_to_mean", "price_ratio_to_max", "snap", "is_event"]
)
CAT_FEATURES = ["dept_id", "cat_id"]
KEY = ["store_id", "item_id"]

def build_features(df, horizon=FORECAST_HORIZON):
    """Vectorised version (much faster and lighter than a per-item loop)."""
    df = df.sort_values(["item_id", "date"]).reset_index(drop=True)
    g = df.groupby("item_id", sort=False)["sales"]
    for l in LAGS:
        df[f"sales_lag_{l}"] = g.shift(l).astype("float32")
    df["_shifted"] = g.shift(horizon)
    gs = df.groupby("item_id", sort=False)["_shifted"]
    for w in WINDOWS:
        df[f"rolling_mean_{w}"] = gs.rolling(w).mean().reset_index(level=0, drop=True).astype("float32")
        df[f"rolling_stdv_{w}"] = gs.rolling(w).std().reset_index(level=0, drop=True).astype("float32")
    df = df.drop(columns="_shifted")
    df["day_of_week"] = df["date"].dt.dayofweek
    df["day_of_month"] = df["date"].dt.day
    df["week_of_year"] = df["date"].dt.isocalendar().week.astype(int)
    df["month"] = df["date"].dt.month
    df["quarter"] = df["date"].dt.quarter
    df["end_of_week"] = (df["day_of_week"] >= 5).astype(int)
    df["is_event"] = df["event_name_1"].notna().astype(int)
    gp = df.groupby("item_id", sort=False)["sell_price"]
    csum = df["sell_price"].fillna(0).groupby(df["item_id"], sort=False).cumsum()
    ccnt = df["sell_price"].notna().groupby(df["item_id"], sort=False).cumsum()
    df["price_ratio_to_mean"] = (df["sell_price"] / (csum / ccnt.replace(0, np.nan))).astype("float32")
    df["price_ratio_to_max"] = (df["sell_price"] / gp.cummax()).astype("float32")
    return df

def to_category(df):
    df = df.copy()
    for c in CAT_FEATURES:
        df[c] = df[c].astype("category")
    return df

def rmsse(y_true, y_pred, hist):
    """Base paper Eq. 1"""
    denom = np.mean(np.diff(hist) ** 2) if len(hist) > 1 else 0
    return np.sqrt(np.mean((y_true - y_pred) ** 2) / (denom if denom > 0 else 1e-9))

def sme(y_true, y_pred, hist):
    """Base paper Eq. 2"""
    denom = np.mean(np.abs(np.diff(hist))) if len(hist) > 1 else 0
    return np.mean(y_true - y_pred) / (denom if denom > 0 else 1e-9)

def suggest_params(trial):
    return {
        "objective": "tweedie",
        "tweedie_variance_power": trial.suggest_float("tweedie_variance_power", 1.01, 1.9),
        "n_estimators": trial.suggest_int("n_estimators", 100, 300),
        "num_leaves": trial.suggest_int("num_leaves", 31, 127),
        "min_data_in_leaf": trial.suggest_int("min_data_in_leaf", 200, 800),
        "learning_rate": trial.suggest_float("learning_rate", 0.015, 0.06),
        "feature_fraction": trial.suggest_float("feature_fraction", 0.4, 0.8),
        "max_bin": trial.suggest_int("max_bin", 100, 150),
        "lambda_l1": trial.suggest_float("lambda_l1", 0.0, 1.5),
        "lambda_l2": trial.suggest_float("lambda_l2", 0.0, 1.5),
    }

FIXED_PARAMS = {"subsample": 0.5, "subsample_freq": 1, "verbosity": -1,
                "seed": RANDOM_SEED, "n_jobs": -1}

def split_train_valid(df):
    """Same rule for every method: the last VALID_DAYS of the training data are used for tuning."""
    cut = df["date"].max() - pd.Timedelta(days=VALID_DAYS - 1)
    return df[df["date"] < cut], df[df["date"] >= cut]

def pruning_callback(trial):
    """Reports validation RMSE to Optuna every PRUNE_CHECK_EVERY rounds so bad trials stop early."""
    def _cb(env):
        if (env.iteration + 1) % PRUNE_CHECK_EVERY:
            return
        for item in env.evaluation_result_list:
            if item[1] == "rmse":
                trial.report(item[2], env.iteration)
                if trial.should_prune():
                    raise optuna.TrialPruned()
    return _cb

def fit_lgb(params, tr, va=None, trial=None):
    model = lgb.LGBMRegressor(**params, **FIXED_PARAMS)
    X_tr = tr[MANUAL_FEATURES + CAT_FEATURES]
    if trial is not None and va is not None:
        model.fit(X_tr, tr["sales"], categorical_feature=CAT_FEATURES,
                  eval_set=[(va[MANUAL_FEATURES + CAT_FEATURES], va["sales"])],
                  eval_metric="rmse", callbacks=[pruning_callback(trial)])
    else:
        model.fit(X_tr, tr["sales"], categorical_feature=CAT_FEATURES)
    return model

def tune_and_train(df, n_trials, sampler, use_pruning=False):
    """Tune on (train, valid) split, then refit the best configuration on all training data."""
    df = to_category(df.dropna(subset=MANUAL_FEATURES + ["sales"]))
    tr, va = split_train_valid(df)
    pruner = MedianPruner(n_startup_trials=5, n_warmup_steps=PRUNE_CHECK_EVERY) if use_pruning else NopPruner()

    def objective(trial):
        params = suggest_params(trial)
        model = fit_lgb(params, tr, va, trial if use_pruning else None)
        pred = model.predict(va[MANUAL_FEATURES + CAT_FEATURES])
        del model; gc.collect()
        return float(np.sqrt(np.mean((va["sales"].values - pred) ** 2)))

    study = optuna.create_study(direction="minimize", sampler=sampler, pruner=pruner)
    study.optimize(objective, n_trials=n_trials, show_progress_bar=False)
    n_pruned = sum(t.state == optuna.trial.TrialState.PRUNED for t in study.trials)
    final_model = fit_lgb(study.best_params, df)
    del tr, va; gc.collect()
    return final_model, study.best_value, study.best_params, n_pruned

PREDS = []   # test-period predictions of every method (saved at the end for analysis / dashboard)

def evaluate_on_test(model, test_df, train_df, method=None):
    """Per product-store RMSSE and SME on the test period (base paper Eq. 1 and 2)."""
    test_df = to_category(test_df.dropna(subset=MANUAL_FEATURES))
    test_df = test_df.assign(pred=model.predict(test_df[MANUAL_FEATURES + CAT_FEATURES]))
    if method:
        PREDS.append(test_df[KEY + ["date", "sales", "pred"]].assign(method=method))
    hist = {k: g["sales"].values for k, g in train_df.sort_values("date").groupby(KEY)}
    rows = []
    for k, g in test_df.groupby(KEY):
        h = hist.get(k)
        if h is None or len(h) < 2:
            continue
        rows.append({"store_id": k[0], "item_id": k[1],
                     "rmsse": rmsse(g["sales"].values, g["pred"].values, h),
                     "sme": sme(g["sales"].values, g["pred"].values, h),
                     "max_pred": float(g["pred"].max()), "max_actual": float(g["sales"].max())})
    return pd.DataFrame(rows)

def result_path(name):
    return f"{OUT_DIR}/{name}.csv"

def log_tuning(row):
    """Append one tuning record to Drive immediately (survives disconnects)."""
    path = result_path("tuning_log")
    pd.DataFrame([row]).to_csv(path, mode="a", header=not os.path.exists(path), index=False)

print("Functions ready. Features used:", len(MANUAL_FEATURES) + len(CAT_FEATURES))


## Step 3 - Build and cache features for each store (done once, reused by every method)

In [ ]:
def feature_paths(store_id):
    return f"{FEATS_STORE_DIR}/{store_id}_train.parquet", f"{FEATS_STORE_DIR}/{store_id}_test.parquet"

for store_id in store_list:
    tr_path, te_path = feature_paths(store_id)
    if os.path.exists(tr_path) and os.path.exists(te_path):
        print(f"{store_id}: features already built, skipping")
        continue
    raw = pd.read_parquet(f"{MERGED_DIR}/{store_id}.parquet")
    # keep TRAIN_WINDOW_DAYS of training data + 200 days so the 180-day rolling features can be computed
    raw = raw[raw["date"] >= test_start - pd.Timedelta(days=TRAIN_WINDOW_DAYS + 200)]
    feats = build_features(raw); del raw; gc.collect()
    train = feats[(feats["date"] < test_start) & (feats["date"] >= test_start - pd.Timedelta(days=TRAIN_WINDOW_DAYS))]
    test = feats[feats["date"] >= test_start]
    train.to_parquet(tr_path, index=False)
    test.to_parquet(te_path, index=False)
    print(f"{store_id}: train {len(train):,} rows, test {len(test):,} rows")
    del feats, train, test; gc.collect()

def load_store(store_id):
    tr_path, te_path = feature_paths(store_id)
    return pd.read_parquet(tr_path), pd.read_parquet(te_path)


## Step 4 - Baseline: base paper DTF (one LightGBM per store, Optuna TPE, 21 trials, no pruning)

In [ ]:
models_for_shap = {}
if os.path.exists(result_path("baseline_dtf_results")):
    baseline_df = pd.read_csv(result_path("baseline_dtf_results"))
    print(f"Baseline already finished - loaded {len(baseline_df):,} products from Drive (not re-trained)")
else:
    baseline_results = []
    for store_id in store_list:
        tr, te = load_store(store_id)
        model, val, params, _ = tune_and_train(tr, N_TRIALS_BASELINE, TPESampler(seed=RANDOM_SEED))
        models_for_shap[("Baseline DTF", store_id)] = model
        res = evaluate_on_test(model, te, tr, method="Base paper DTF (replicated)")
        baseline_results.append(res)
        log_tuning({"method": "Baseline DTF", "group": store_id, "valid_rmse": val, **params})
        print(f"{store_id}: median test RMSSE {res['rmsse'].median():.4f}")
        del tr, te; gc.collect()
    baseline_df = pd.concat(baseline_results, ignore_index=True)
    baseline_df.to_csv(result_path("baseline_dtf_results"), index=False)
print(f"BASELINE DTF - median RMSSE {baseline_df['rmsse'].median():.4f}  (final mean is computed in Step 8)")


## Step 5 - Optimization 1: CMA-ES sampler + median pruning (same model, larger smarter search)

In [ ]:
if os.path.exists(result_path("optimization1_cmaes_results")):
    cmaes_df = pd.read_csv(result_path("optimization1_cmaes_results"))
    print(f"Optimization 1 already finished - loaded {len(cmaes_df):,} products from Drive")
else:
    cmaes_results = []
    for store_id in store_list:
        tr, te = load_store(store_id)
        model, val, params, n_pruned = tune_and_train(tr, N_TRIALS_OPTIMIZED,
                                                       CmaEsSampler(seed=RANDOM_SEED), use_pruning=True)
        models_for_shap[("Opt1 CMA-ES + pruning", store_id)] = model
        res = evaluate_on_test(model, te, tr, method="Opt 1: CMA-ES + pruning")
        cmaes_results.append(res)
        log_tuning({"method": "Opt1 CMA-ES + pruning", "group": store_id, "valid_rmse": val,
                    "trials_pruned": n_pruned, **params})
        print(f"{store_id}: median test RMSSE {res['rmsse'].median():.4f}  ({n_pruned}/{N_TRIALS_OPTIMIZED} trials pruned)")
        del tr, te; gc.collect()
    cmaes_df = pd.concat(cmaes_results, ignore_index=True)
    cmaes_df.to_csv(result_path("optimization1_cmaes_results"), index=False)
print(f"OPT 1 (CMA-ES + pruning) - median RMSSE {cmaes_df['rmsse'].median():.4f} | baseline {baseline_df['rmsse'].median():.4f}")


## Step 6 - Optimization 2: demand-pattern pooling (one model per ADI/CV² category instead of per store)

In [ ]:
def classify(sales):
    """Syntetos et al. (2005) classification, as used in the base paper."""
    nz = sales[sales > 0]
    if len(nz) < 2:
        return np.nan, np.nan, "lumpy"   # very sparse items are treated as lumpy
    adi = len(sales) / len(nz)
    cv2 = (nz.std() / nz.mean()) ** 2
    if adi < 4/3:
        return adi, cv2, "smooth" if cv2 < 0.49 else "erratic"
    return adi, cv2, "intermittent" if cv2 < 0.49 else "lumpy"

rows = []
for store_id in store_list:
    tr = pd.read_parquet(feature_paths(store_id)[0], columns=["store_id", "item_id", "sales"])
    for (s, i), g in tr.groupby(KEY):
        adi, cv2, cat = classify(g["sales"].values)
        rows.append({"store_id": s, "item_id": i, "ADI": adi, "CV2": cv2, "category": cat})
    del tr; gc.collect()
class_df = pd.DataFrame(rows)
class_df.to_csv(f"{OUT_DIR}/demand_pattern_classification.csv", index=False)
print(class_df["category"].value_counts())

def load_category_data():
    """All products of the selected stores, regrouped by demand category (every product is kept)."""
    trs, tes = [], []
    for store_id in store_list:
        tr, te = load_store(store_id)
        trs.append(tr); tes.append(te)
    tr_all = pd.concat(trs, ignore_index=True).merge(class_df[KEY + ["category"]], on=KEY)
    te_all = pd.concat(tes, ignore_index=True).merge(class_df[KEY + ["category"]], on=KEY)
    del trs, tes; gc.collect()
    return tr_all, te_all

if os.path.exists(result_path("optimization2_pooling_results")):
    pooling_df = pd.read_csv(result_path("optimization2_pooling_results"))
    print(f"Optimization 2 already finished - loaded {len(pooling_df):,} products from Drive")
else:
    tr_all, te_all = load_category_data()
    pooling_results = []
    for cat in sorted(class_df["category"].unique()):
        tr_c, te_c = tr_all[tr_all["category"] == cat], te_all[te_all["category"] == cat]
        model, val, params, _ = tune_and_train(tr_c, N_TRIALS_BASELINE, TPESampler(seed=RANDOM_SEED))
        res = evaluate_on_test(model, te_c, tr_c, method="Opt 2: demand-pattern pooling")
        pooling_results.append(res)
        log_tuning({"method": "Opt2 demand pooling", "group": cat, "valid_rmse": val, **params})
        print(f"{cat}: {tr_c[KEY].drop_duplicates().shape[0]} products, median test RMSSE {res['rmsse'].median():.4f}")
        del model; gc.collect()
    del tr_all, te_all; gc.collect()
    pooling_df = pd.concat(pooling_results, ignore_index=True)
    pooling_df.to_csv(result_path("optimization2_pooling_results"), index=False)
print(f"OPT 2 (demand pooling) - median RMSSE {pooling_df['rmsse'].median():.4f} | baseline {baseline_df['rmsse'].median():.4f}")


## Step 7 - Optimization 3: joint optimization of pooling strategy + hyperparameters

For each pooling strategy, CMA-ES searches the hyperparameters, and every trial is scored on the **same validation rows (all products)**.
The strategy/hyperparameter pair with the lowest validation error wins and is then evaluated on the test period.

In [ ]:
if os.path.exists(result_path("optimization3_joint_results")):
    joint_df = pd.read_csv(result_path("optimization3_joint_results"))
    print(f"Optimization 3 already finished - loaded {len(joint_df):,} products from Drive")
else:
    tr_all, te_all = load_category_data()
    tr_all = to_category(tr_all.dropna(subset=MANUAL_FEATURES + ["sales"]))
    STRATEGIES = {"per_store": "store_id", "per_demand_category": "category"}

    def joint_search(group_col):
        groups = sorted(tr_all[group_col].unique())
        splits = {g: split_train_valid(tr_all[tr_all[group_col] == g]) for g in groups}

        def objective(trial):
            params = suggest_params(trial)
            se, n = 0.0, 0
            for step, g in enumerate(groups):
                tr, va = splits[g]
                model = fit_lgb(params, tr)
                pred = model.predict(va[MANUAL_FEATURES + CAT_FEATURES])
                se += float(np.sum((va["sales"].values - pred) ** 2)); n += len(va)
                del model; gc.collect()
                trial.report(np.sqrt(se / n), step)       # running error -> lets the pruner stop bad trials
                if trial.should_prune():
                    raise optuna.TrialPruned()
            return float(np.sqrt(se / n))

        study = optuna.create_study(direction="minimize", sampler=CmaEsSampler(seed=RANDOM_SEED),
                                    pruner=MedianPruner(n_startup_trials=5))
        study.optimize(objective, n_trials=N_TRIALS_JOINT_PER_STRATEGY, show_progress_bar=False)
        return study

    joint_summary = []
    for name, col in STRATEGIES.items():
        study = joint_search(col)
        joint_summary.append({"strategy": name, "best_valid_rmse": study.best_value, **study.best_params})
        print(f"{name}: best validation RMSE {study.best_value:.4f}")

    joint_summary_df = pd.DataFrame(joint_summary).sort_values("best_valid_rmse")
    joint_summary_df.to_csv(f"{OUT_DIR}/optimization3_joint_search.csv", index=False)
    best = joint_summary_df.iloc[0]
    best_strategy, best_col = best["strategy"], STRATEGIES[best["strategy"]]
    best_params = {k: (int(best[k]) if k in ["n_estimators", "num_leaves", "min_data_in_leaf", "max_bin"] else float(best[k]))
                   for k in ["tweedie_variance_power", "n_estimators", "num_leaves", "min_data_in_leaf",
                             "learning_rate", "feature_fraction", "max_bin", "lambda_l1", "lambda_l2"]}
    best_params["objective"] = "tweedie"
    print(f"\nWinning strategy: {best_strategy}")

    joint_results = []
    for g in sorted(tr_all[best_col].unique()):
        tr_g, te_g = tr_all[tr_all[best_col] == g], te_all[te_all[best_col] == g]
        model = fit_lgb(best_params, tr_g)
        joint_results.append(evaluate_on_test(model, te_g, tr_g, method="Opt 3: joint optimization"))
        del model; gc.collect()
    del tr_all, te_all; gc.collect()

    joint_df = pd.concat(joint_results, ignore_index=True)
    joint_df.to_csv(result_path("optimization3_joint_results"), index=False)
print(f"OPT 3 (joint) - median RMSSE {joint_df['rmsse'].median():.4f} | baseline {baseline_df['rmsse'].median():.4f}")


## Step 8 - Comparison on exactly the same products

In [ ]:
results = {"Base paper DTF (replicated)": baseline_df,
           "Opt 1: CMA-ES + pruning": cmaes_df,
           "Opt 2: demand-pattern pooling": pooling_df,
           "Opt 3: joint optimization": joint_df}

# RMSSE is undefined for products with no sales change during training (denominator = 0).
# Find those products and leave them out of every method's score.
scorable = set()
excluded = []
for s in store_list:
    tr = pd.read_parquet(feature_paths(s)[0], columns=["store_id", "item_id", "date", "sales"]).sort_values("date")
    for k, g in tr.groupby(KEY):
        if np.mean(np.diff(g["sales"].values) ** 2) > 0:
            scorable.add(k)
        else:
            excluded.append({"store_id": k[0], "item_id": k[1]})
    del tr; gc.collect()
pd.DataFrame(excluded).to_csv(f"{OUT_DIR}/excluded_no_sales_variation.csv", index=False)
print(f"Excluded {len(excluded)} products with no sales variation in the training window")

common = set(scorable)
for df in results.values():
    common &= set(map(tuple, df[KEY].values))
print(f"Products scored by every method: {len(common):,}")

def on_common(df):
    return df[[k in common for k in map(tuple, df[KEY].values)]]

base_rmsse = on_common(baseline_df)["rmsse"].mean()
summary = []
for name, df in results.items():
    d = on_common(df)
    summary.append({"method": name, "products": len(d),
                    "mean_rmsse": d["rmsse"].mean(), "median_rmsse": d["rmsse"].median(),
                    "mean_sme": d["sme"].mean(),
                    "improvement_vs_baseline_%": 100 * (base_rmsse - d["rmsse"].mean()) / base_rmsse})
summary_df = pd.DataFrame(summary)
summary_df.to_csv(f"{OUT_DIR}/comparison_summary.csv", index=False)
print(summary_df.round(4).to_string(index=False))

by_cat = []
for name, df in results.items():
    d = on_common(df).merge(class_df[KEY + ["category"]], on=KEY)
    for cat, g in d.groupby("category"):
        by_cat.append({"method": name, "category": cat, "products": len(g), "mean_rmsse": g["rmsse"].mean()})
pd.DataFrame(by_cat).to_csv(f"{OUT_DIR}/comparison_by_category.csv", index=False)

by_store = []
for name, df in results.items():
    for s, g in on_common(df).groupby("store_id"):
        by_store.append({"method": name, "store_id": s, "mean_rmsse": g["rmsse"].mean()})
pd.DataFrame(by_store).to_csv(f"{OUT_DIR}/comparison_by_store.csv", index=False)
if PREDS:
    pd.concat(PREDS, ignore_index=True).to_csv(f"{OUT_DIR}/test_predictions_{pd.Timestamp.now():%H%M}.csv.gz", index=False)
    print("Saved test-period predictions")


## Step 9 - SHAP feature importance (baseline model, first store)

In [ ]:
import shap
store_id = store_list[0]
tr, _ = load_store(store_id)
tr = to_category(tr.dropna(subset=MANUAL_FEATURES + ["sales"]))
model, label = None, None
for m in ["Baseline DTF", "Opt1 CMA-ES + pruning"]:
    if (m, store_id) in models_for_shap:
        model, label = models_for_shap[(m, store_id)], m
        break
if model is None:
    log = pd.read_csv(result_path("tuning_log"))
    row = log[(log["method"] == "Baseline DTF") & (log["group"] == store_id)]
    if len(row) == 0:
        row = log[log["group"] == store_id]
    row = row.iloc[-1]
    params = {k: (int(row[k]) if k in ["n_estimators", "num_leaves", "min_data_in_leaf", "max_bin"] else float(row[k]))
              for k in ["tweedie_variance_power", "n_estimators", "num_leaves", "min_data_in_leaf",
                        "learning_rate", "feature_fraction", "max_bin", "lambda_l1", "lambda_l2"]}
    params["objective"] = "tweedie"
    model, label = fit_lgb(params, tr), row["method"] + " (refit with saved best settings)"
print("SHAP explains:", label, "-", store_id)
X = tr[MANUAL_FEATURES + CAT_FEATURES].sample(min(3000, len(tr)), random_state=RANDOM_SEED)
sv = shap.TreeExplainer(model).shap_values(X)
shap_df = pd.DataFrame({"feature": X.columns, "mean_abs_shap": np.abs(sv).mean(axis=0)}) \
            .sort_values("mean_abs_shap", ascending=False)
shap_df["share_%"] = 100 * shap_df["mean_abs_shap"] / shap_df["mean_abs_shap"].sum()
shap_df["model"] = label
shap_df.to_csv(f"{OUT_DIR}/shap_importance.csv", index=False)
print(shap_df.head(15).round(4).to_string(index=False))


## Step 10 - Export: one zip with every result CSV

In [ ]:
from google.colab import files
csvs = sorted(f for f in os.listdir(OUT_DIR) if f.endswith(".csv"))
zip_dir = f"{OUT_DIR}/_zip"
os.makedirs(zip_dir, exist_ok=True)
for f in csvs:
    shutil.copy(f"{OUT_DIR}/{f}", zip_dir)
zip_path = shutil.make_archive(f"{OUT_DIR}/m5_results_csvs", "zip", zip_dir)
shutil.rmtree(zip_dir)
print("Result files:", csvs)
files.download(zip_path)
